# c21-2026: low birth weight classification, temporal validation

1. No GPU needed.
2. **Runtime → Run all.** The pipeline runs once. Finished stages are skipped and a stage that failed half way resumes from its last checkpoint; tick `FORCE` in cell 3 to redo everything.
3. **If anything fails**, the last cell prints `outputs/logs/errors.log`, the only file to read. `outputs/logs/status.json` says which stage failed.

After the agent syncs new code, re-run cell 1, then cell 3. How to use this notebook and the rules for changing it: `COLAB.md` in the repository. Generated from `experiments/colab.yaml` by `scripts/paper_notebook.py`; do not edit it by hand.

In [ ]:
#@title 1. Load the latest code from Drive (re-run after every agent sync)
import importlib.util
import os
import pathlib
import shutil
import sys
from datetime import datetime

from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = pathlib.Path("/content/drive/MyDrive/c21-2026")
CODE_DIR = PROJECT_DIR / "code"
WORK_DIR = pathlib.Path("/content/c21-2026/experiments")
os.environ["DATA_DIR"] = str(PROJECT_DIR / "data")
os.environ["OUTPUT_DIR"] = str(PROJECT_DIR / "outputs")
if not (CODE_DIR / "_colab_runtime.py").is_file():
    raise RuntimeError(f"{CODE_DIR} has no pipeline code: ask the agent to sync it")

# Loaded straight from Drive, so the error hook covers the rest of this cell.
# No bytecode: it would be written into Drive code/ as __pycache__.
sys.dont_write_bytecode = True
_spec = importlib.util.spec_from_file_location("_colab_runtime", CODE_DIR / "_colab_runtime.py")
rt = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(rt)
sys.modules["_colab_runtime"] = rt
LOG = rt.RunLog(PROJECT_DIR / "outputs" / "logs")
LOG.begin_session(note="notebook")
rt.install_cell_error_hook(LOG)

shutil.rmtree(WORK_DIR, ignore_errors=True)
shutil.copytree(CODE_DIR, WORK_DIR, ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
for _name, _mod in list(sys.modules.items()):  # forget modules from an older copy
    if str(getattr(_mod, "__file__", "") or "").startswith(str(WORK_DIR)):
        del sys.modules[_name]
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)
SPEC = rt.load_spec(WORK_DIR)

_files = sorted(p for p in WORK_DIR.rglob("*") if p.is_file() and "__pycache__" not in p.parts)
_newest = max(_files, key=lambda p: p.stat().st_mtime)
print(f"[ok] {len(_files)} files copied from {CODE_DIR}")
print(f"[ok] newest: {_newest.name} "
      f"({datetime.fromtimestamp(_newest.stat().st_mtime):%Y-%m-%d %H:%M} UTC)")
print(f"[ok] errors of this session go to {LOG.errors}")

In [ ]:
#@title 2. Check the runtime (GPU, dependencies, data, probes)
rt.preflight(SPEC, WORK_DIR, LOG)

In [ ]:
#@title 3. Run the pipeline (the only cell that runs it)
MODE = "full"  #@param ["full", "smoke"]
FORCE = False  #@param {type:"boolean"}

EXIT_CODE = rt.run_pipeline(SPEC, WORK_DIR, LOG, mode=MODE, force=FORCE)

In [ ]:
#@title 4. Run one step (manual: nothing runs under Run all)
def step(*stages, mode="full"):
    """Run only these stages, always re-running them: step("03"), step("07", "07b")."""
    return rt.run_pipeline(SPEC, WORK_DIR, LOG, mode=mode, only=list(stages))


def step_range(first, last, mode="full"):
    """Run an inclusive range, skipping stages that are still current."""
    return rt.run_pipeline(SPEC, WORK_DIR, LOG, mode=mode, extra=["--from", first, "--to", last])


# Uncomment one line, then run this cell:
# step("03")
# step("06b")
# step_range("07", "10")

In [ ]:
#@title 5. Results and errors
rt.run_report(SPEC, WORK_DIR, LOG)
rt.raise_if_errors(LOG)